In [139]:
import re
from typing import List
import time

class DailyLogActivities:
    """A Class to hold the information for a Daily Log Entry"""
    input_text: str
    processed_text: str

    def __init__(self, input_text: str):
        """Creates a new Daily Log Entry from the input_text str"""
        self.input_text = input_text
        self._getTime()
        self._getProcessedText()

    def _getTime(self):
        """Get the times from a input_text string"""
        activity_prefix = ">@"
        extra_prefix = "@<"
        time_pattern = r"(?<=\b)([0-9]{3,4}|[0-9]{1,2}:[0-9]{1,2})(?=\b)"
        matches = re.findall(time_pattern, self.input_text)
        for ind, match in enumerate(matches):
            match = match.rjust(4, "0")
            #matches[ind] = datetime.datetime.strptime(match, "%H%M")
            matches[ind] = time.strptime(match, "%H%M")
        self.start_time = matches

    def _getProcessedText(self):
        """Get the text without the times"""
        time_pattern = r"(?<=\b)[0-9]{3,4}|[0-9]{1,2}:[0-9]{1,2}(?=\b)"
        self.processed_text = "".join(re.split(time_pattern, self.input_text)).strip()

Classes for the parse items

In [140]:
class ParseClass:
    """
    """
    def __init__(self, parse_type: str):
        self.parse_type: str = parse_type

In [141]:
import datetime

class EntryTime(ParseClass):
    """
    """
    def __init__(self, start_time: datetime.datetime):
        super().__init__(parse_type = "Time")
        self.start_time: datetime.datetime = start_time

In [142]:
class EntryActivity(ParseClass):
    """
    """
    def __init__(self, activity: str):
        super().__init__(parse_type = "Activity")
        self.activity: str = activity

In [143]:
class EntryComment(ParseClass):
    """
    """
    def __init__(self, comment: str):
        super().__init__(parse_type = "Comment")
        self.comment: str = comment

Classes for the parse search terms

In [144]:
#from __future__ import annotations
from abc import ABC, abstractmethod

class ParseSearchTerm(ABC):
    """
    """
    def __init__(self, search_term: str):
        self.search_term: str = search_term

    @abstractmethod
    def process(self, check_char: str) -> ParseClass | None:
        pass

In [145]:
class ParseSearchTermFinal(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str, return_class: ParseClass):
        super().__init__(search_term = search_term)
        self.return_class = return_class

    def process(self, check_char: str) -> ParseClass | None:
        """
        """
        if(check_char == self.search_term):
            return self.return_class
        return None

In [146]:
class ParseSearchTermIntermediary(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str):
        super().__init__(search_term = search_term)
        self.next_phrases: list[ParseSearchTerm] = []

    def process(self, check_char: str) -> ParseClass | None:
        """
        """
        if check_char == self.search_term:
            for next_phrase in self.next_phrases:
                temp_result = next_phrase.process(check_char = check_char)
                if temp_result is not None:
                    return temp_result
        return None
    
    def addNextTerm(self, next_phrase: ParseSearchTerm):
        """
        """
        if isinstance(next_phrase, ParseSearchTerm):
            self.next_phrases.append(next_phrase)
        else:
            raise ValueError("ParseSearchTermIntermediary.addNextTerm input is not a ParseSearchTerm")

In [147]:
class ParseSearchTerms:
    """
    """
    def __init__(self):
        self.search_terms: list[ParseSearchTerm] = []

    def addSearchTerm(self, search_term: ParseSearchTerm):
        """
        """
        if isinstance(search_term, ParseSearchTerm):
            self.search_terms.append(search_term)
        else:
            raise ValueError("ParseSearchTerms.addSearchTerm input is not a ParseSearchTerm")
        
    def getSearchTerms(self, check_char: str):
        for search_term in self.search_terms:
            temp_result = search_term.process(check_char = check_char)
            print(temp_result)

Classes for the parse search

In [148]:
class ParseSearch(ParseSearchTerm):
    """
    """
    def __init__(self, search_term: str):
        super().__init__(search_term = search_term)
        self.parser: ParseSearch = ParseSearch()

In [149]:
class Parser:
    """
    """
    def __init__(self, parser_options: ParseSearchTerms):
        self.entry_tree = []
        self.parser_options = parser_options

    def _checkMatchingCharacter(self, check_char: str):
        self.parser_options.getSearchTerms(check_char = check_char)

    def search(self, source: str):
        """
        """
        cur_mode = None
        for char in source:
            print(char)

Parser Options

In [150]:
at_at_mode = ParseSearchTermFinal("@", EntryTime)
at_less_mode = ParseSearchTermFinal("<", EntryComment)
at_mode = ParseSearchTermIntermediary("@")
at_mode.addNextTerm(at_at_mode)
at_mode.addNextTerm(at_less_mode)

greater_at_mode = ParseSearchTermFinal("@", EntryTime)
greater_mode = ParseSearchTermIntermediary(">")
greater_mode.addNextTerm(greater_at_mode)

parser_options = ParseSearchTerms()
parser_options.addSearchTerm(greater_mode)
parser_options.addSearchTerm(at_mode)

Test Parser

In [151]:
test_parser = Parser(parser_options = parser_options)

test_parser._checkMatchingCharacter("@")

#test_parser.search("2026-04-24 07:29 @@ Bathroom >@ Reading Berserk @< Cool Chapter >@ @< Chapter 176")

None
<class '__main__.EntryTime'>


Need to modify the check to take the entire current string and to keep iterating instead of using the same character for each next...